In [1]:
import os
import yaml

# Dataset Fetching

In [2]:
yaml_path = r"C:\Users\Himanshu\Downloads\yolo_dataset\dataset.yaml"

with open(yaml_path, "r") as f:
    data = yaml.safe_load(f)

print("Classes:", data["names"], "nc:", {len(data["names"])})
print("Train images path:", data["train"])
print("Validation images path:", data["val"])


Classes: ['UAV'] nc: {1}
Train images path: images/train
Validation images path: images/valid


#  Data Preprocessing

In [3]:
import cv2
import glob
from tqdm import tqdm

Finding Corrupted Dataset

In [4]:
img_dir=data["train"]

all_imgs=glob.glob(os.path.join(img_dir,"*.jpg", "*.jpeg", "*.png"))

bad_imgs=[]

for img_pth in tqdm(all_imgs):
    try:
        img=cv2.imread(img_pth)
        if img is None:
            bad_imgs.append(img_pth)
    except:
        bad_imgs.append(img_pth)
print("Corrupt:", bad_imgs)



0it [00:00, ?it/s]

Corrupt: []


# Data Augmentation

In [6]:
import albumentations  as A
from albumentations.pytorch import ToTensorV2
import numpy as np

ModuleNotFoundError: No module named 'albumentations.augmentations.pixel.transforms'

In [ ]:
transform = A.Compose([
    A.RandomCrop(width=1024, height=1024),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.Rotate(limit=15, border_mode=0, p=0.5),
    A.RandomBrightnessContrast(p=0.5),
    A.HueSaturationValue(p=0.3),
    A.MotionBlur(blur_limit=5, p=0.2),
    A.RandomFog(p=0.2),
    A.RandomRain(p=0.2),
    A.GaussNoise(p=0.3),
    ],
    bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']),
    mask_params=A.MaskParams()
    )

In [ ]:
img = cv2.imread(all_imgs[0])
mask = cv2.imread(all_masks[0], cv2.IMREAD_GRAYSCALE)  # binary mask of UAV from polygon

aug = transform(image=img, masks=[mask])
aug_img = aug["image"]
aug_mask = aug["masks"][0]

# Show augmented image
cv2.imshow("Augmented Image", aug_img)
cv2.imshow("Augmented Mask", aug_mask)
cv2.waitKey(0)
cv2.destroyAllWindows()

CUDA available True


In [2]:
# infer_and_clean.py
import os
import cv2
import numpy as np
from ultralytics import YOLO

# ----- CONFIG -----
MODEL_PATH = r"D:\Himanshu_ML\TE_UAV Projects\TE_1 UAV MRO\UAV_Parts\runs\segment\Body_Segment\weights\best.pt"
IMG_PATH = r"D:\Himanshu_ML\TE_UAV Data\Tracking Dataset\fixed wing UAV\train\images\2022-06-13-337-_png_jpg.rf.c37461d3fd3f818132a1b52fc4d879f6.jpg"
CLASS_NAMES = ['UAV']   # ensure order matches your training names
COLORS = [(60, 100, 255)]  # BGR color for mask overlay (can add more)
INFER_IMG_SIZE = 1024   # larger -> more accurate masks but slower
CONF_THRESH = 0.25
IOU_THRESH = 0.45
MIN_AREA_KEEP = 2000    # min area (pixels) for kept object

# ----- UTIL -----
def poly_to_mask(poly_xy, H, W):
    """Convert polygon (list of [x,y]) to binary mask HxW (uint8 0/255)."""
    pts = np.array(poly_xy, dtype=np.int32).reshape((-1, 2))
    mask = np.zeros((H, W), dtype=np.uint8)
    cv2.fillPoly(mask, [pts], 255)
    return mask

def clean_mask(mask, close_k=9, open_k=7):
    """Morphologically close then open to remove noise and fill holes."""
    k_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (close_k, close_k))
    k_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (open_k, open_k))
    m = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k_close)
    m = cv2.morphologyEx(m, cv2.MORPH_OPEN, k_open)
    return m

def keep_largest_component(mask, min_area=1000):
    """Keep only the largest connected component if it exceeds min_area."""
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    if num_labels <= 1:
        return np.zeros_like(mask)
    areas = stats[1:, cv2.CC_STAT_AREA]
    largest_idx = np.argmax(areas) + 1
    if areas[largest_idx - 1] < min_area:
        return np.zeros_like(mask)
    out = np.zeros_like(mask)
    out[labels == largest_idx] = 255
    return out

def mask_to_polygons(mask, epsilon=5.0):
    """Return list of polygons (list of points) from a binary mask."""
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    polys = []
    for c in cnts:
        if cv2.contourArea(c) < 10: 
            continue
        # approx to reduce vertex count
        peri = cv2.arcLength(c, True)
        approx = cv2.approxPolyDP(c, epsilon, True)
        pts = approx.squeeze().tolist()
        if len(pts) >= 3:
            polys.append(pts if isinstance(pts[0], list) else [pts.tolist()])
    return polys

# ----- LOAD MODEL -----
model = YOLO(MODEL_PATH)

# ----- LOAD IMAGE -----
img = cv2.imread(IMG_PATH)
if img is None:
    raise FileNotFoundError(f"Image not found: {IMG_PATH}")
H, W = img.shape[:2]

# ----- RUN INFERENCE (TTA optional through augment=True) -----
results = model(img, imgsz=INFER_IMG_SIZE, conf=CONF_THRESH, iou=IOU_THRESH, augment=True)[0]

# ----- COLLECT & COMBINE POLYGONS INTO ONE MASK -----
pred_polys = []
if results.masks is not None:
    # results.masks.xy is a list of polygons per detected instance (pixel coords)
    for seg, box in zip(results.masks.xy, results.boxes):
        conf = float(box.conf[0])
        if conf < CONF_THRESH:
            continue
        poly = seg.tolist()  # list of [x,y]
        pred_polys.append(poly)

if len(pred_polys) == 0:
    print("No masks predicted above confidence threshold.")
    combined_mask = np.zeros((H, W), dtype=np.uint8)
else:
    combined_mask = np.zeros((H, W), dtype=np.uint8)
    for poly in pred_polys:
        m = poly_to_mask(poly, H, W)
        combined_mask = cv2.bitwise_or(combined_mask, m)

    # clean + keep largest
    combined_mask = clean_mask(combined_mask, close_k=15, open_k=9)
    combined_mask = keep_largest_component(combined_mask, min_area=MIN_AREA_KEEP)

# ----- convert final mask to polygons (for export/display) -----
final_polys = mask_to_polygons(combined_mask, epsilon=3.0)

# ----- VISUALIZE: overlay + bounding box + label -----
overlay = img.copy()
alpha = 0.45
color = COLORS[0]  # BGR

if combined_mask.sum() > 0:
    # draw filled mask
    pts_for_draw = [np.array(p, dtype=np.int32).reshape(-1,2) for p in final_polys]
    if pts_for_draw:
        cv2.fillPoly(overlay, pts_for_draw, color)
    img_display = cv2.addWeighted(overlay, alpha, img, 1-alpha, 0)
else:
    img_display = img.copy()

# draw bounding boxes around final mask contours
cnts, _ = cv2.findContours(combined_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
for c in cnts:
    x,y,w,h = cv2.boundingRect(c)
    cv2.rectangle(img_display, (x,y), (x+w,y+h), (0,255,0), 2)
    # label
    label_text = f"UAV"
    cv2.putText(img_display, label_text, (x, max(y-8, 12)), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,255), 2, cv2.LINE_AA)

# Show at original resolution
win = "Detection Result"
cv2.namedWindow(win, cv2.WINDOW_NORMAL)
cv2.imshow(win, img_display)
cv2.resizeWindow(win, W, H)
cv2.waitKey(0)
cv2.destroyAllWindows()

# Optionally: export final polygons (list of points) -> for Label Studio ml backend
print("Final polygons count:", len(final_polys))
# final_polys is ready to be encoded to Label-Studio polygon format if needed



WARNING Model does not support 'augment=True', reverting to single-scale prediction.
0: 1024x1024 (no detections), 44.4ms
Speed: 9.2ms preprocess, 44.4ms inference, 19.8ms postprocess per image at shape (1, 3, 1024, 1024)
No masks predicted above confidence threshold.


error: OpenCV(4.12.0) D:\a\opencv-python\opencv-python\opencv\modules\highgui\src\window.cpp:1284: error: (-2:Unspecified error) The function is not implemented. Rebuild the library with Windows, GTK+ 2.x or Cocoa support. If you are on Ubuntu or Debian, install libgtk2.0-dev and pkg-config, then re-run cmake or configure script in function 'cvNamedWindow'


In [ ]:

from ultralytics import YOLO
from ultralytics.data import build_dataloader

model = YOLO("yolo11l-seg.pt")

# Train with Albumentations
results = model.train(
    data="C:/Users/Himanshu/Downloads/yolo_dataset/dataset.yaml",
    epochs=150,
    imgsz=1024,
    batch=8,
    name="Body_Segment_Albument",
    device=0,
    rect=False,
    amp=True,
    lr0=0.001,
    optimizer="AdamW",
    cos_lr=True,
    warmup_epochs=5,
    weight_decay=0.0005,
    augment=True,
    # ✅ use Albumentations
    transforms=seg_transform
)


In [2]:
import os
import torch
from ultralytics import YOLO
import cv2
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"
print("CUDA available", torch.cuda.is_available())

AttributeError: module 'cv2' has no attribute 'imshow'

In [ ]:
# just load weights
model = YOLO(r"D:\Himanshu_ML\TE_UAV Projects\TE_1 UAV MRO\YOLO_Model\yolo11l-seg.pt")


In [ ]:

model.train(
    
    data=r"C:\Users\Himanshu\Downloads\yolo_dataset\dataset.yaml", 
    epochs=120,
    imgsz=1024,
    batch=8,
    workers=0,                # reduce if GPU OOM
    name="Body_Segment",
    device=0,



    optimizer="AdamW",
    lr0=0.001,
    cos_lr=True, 
    warmup_epochs=5, 
    weight_decay=0.0005,

    dropout=0.1,
    mosaic=0.5,
    degrees=15,
    perspective=0.001,
    translate=0.1,
    scale=0.2,
    flipud=0.1,
    fliplr=0.5,

    amp=True,                   
)

New https://pypi.org/project/ultralytics/8.3.186 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.155  Python-3.12.7 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 2080 Ti, 11264MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=C:\Users\Himanshu\Downloads\yolo_dataset\dataset.yaml, degrees=15, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.1, dynamic=False, embed=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.1, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=D:\Himanshu_ML\TE_UAV Projects\TE_1 UAV MRO\YOLO_Model\yolo11l-seg.pt, momentu

train: Scanning C:\Users\Himanshu\Downloads\yolo_dataset\labels\train.cache... 300 images, 0 backgrounds, 3 corrupt: 100%|██████████| 300/300 [00:00<?, ?it/s]

train: C:\Users\Himanshu\Downloads\yolo_dataset\images\train\03449856-IMG_20250812_113052012_BURST142.jpg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.0005]
train: C:\Users\Himanshu\Downloads\yolo_dataset\images\train\65d700d6-IMG_20250812_112939886_BURST091.jpg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.0003]
train: C:\Users\Himanshu\Downloads\yolo_dataset\images\train\69366d56-IMG_20250812_112616803_BURST084.jpg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.0005]
val: Fast image access  (ping: 0.30.0 ms, read: 376.038.2 MB/s, size: 3495.7 KB)



val: Scanning C:\Users\Himanshu\Downloads\yolo_dataset\labels\valid.cache... 75 images, 0 backgrounds, 2 corrupt: 100%|██████████| 75/75 [00:00<?, ?it/s]

val: C:\Users\Himanshu\Downloads\yolo_dataset\images\valid\1ecda20c-IMG_20250812_112616803_BURST078.jpg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.0005]
val: C:\Users\Himanshu\Downloads\yolo_dataset\images\valid\f63d079d-IMG_20250812_113052012_BURST137.jpg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.0003]


...

                   all         73         73      0.985      0.973      0.986      0.886      0.999      0.986      0.994      0.888



150 epochs completed in 41.317 hours.
Optimizer stripped from runs\segment\Body_Segment\weights\last.pt, 55.9MB
Optimizer stripped from runs\segment\Body_Segment\weights\best.pt, 55.9MB

Validating runs\segment\Body_Segment\weights\best.pt...
Ultralytics 8.3.155  Python-3.12.7 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 2080 Ti, 11264MiB)
YOLO11l-seg summary (fused): 203 layers, 27,585,363 parameters, 0 gradients, 141.9 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100%|██████████| 5/5 [00:07<00:00,  1.60s/it]


                   all         73         73      0.985      0.973      0.986      0.886      0.999      0.986      0.994      0.887
Speed: 0.5ms preprocess, 18.2ms inference, 0.0ms loss, 1.8ms postprocess per image
Results saved to runs\segment\Body_Segment


ultralytics.utils.metrics.SegmentMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000002A2850A15E0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)', 'Precision-Recall(M)', 'F1-Confidence(M)', 'Precision-Confidence(M)', 'Recall-Confidence(M)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.0410

In [3]:
import cv2

In [4]:
import cv2
import numpy as np
from ultralytics import YOLO
import torch
import requests
from PIL import Image
from io import BytesIO
import re
import os

# ======================
# Function: Convert Google Drive link to direct download
# ======================
def get_direct_image_url(url):
    match = re.search(r'/d/([a-zA-Z0-9_-]+)', url)
    if match:
        file_id = match.group(1)
        return f"https://drive.google.com/uc?export=download&id={file_id}"
    return url

# ======================
# Check CUDA availability
# ======================
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))

# ======================
# Load YOLO model (Stage 1 UAV segmentation OR Stage 2 defect detection)
# ======================
model_path= r"D:\Himanshu_ML\TE_UAV Projects\TE_1 UAV MRO\UAV_Parts\runs\segment\Body_Segment\weights\best.pt"
model = YOLO(model_path)
model.to("cuda" if torch.cuda.is_available() else "cpu")

# ======================
# Class names (CHANGE according to training)
# ======================
classNames = ['UAV']   # <-- Stage 1 (whole UAV detection)
# classNames = ['Crack', 'Dent']  # <-- Stage 2 defects
colors = [(255, 0, 0), (0, 255, 0), (0, 0, 255)]  # extend if needed

# ======================
# Image URL or local path
# ======================
url=r"D:\Himanshu_ML\TE_UAV Data\Tracking Dataset\fixed wing UAV\train\images\2022-06-13-337-_png_jpg.rf.c37461d3fd3f818132a1b52fc4d879f6.jpg"
url = get_direct_image_url(url)

# ======================
# Load Image
# ======================
img = None
if url.startswith("http"):
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        image_array = np.asarray(bytearray(response.content), dtype=np.uint8)
        img = cv2.imdecode(image_array, cv2.IMREAD_COLOR)
        if img is None:
            pil_img = Image.open(BytesIO(response.content)).convert("RGB")
            img = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
    except Exception as e:
        print(f"❌ Error loading image from URL: {e}")
else:
    if os.path.exists(url):
        img = cv2.imread(url)
    else:
        print(f"❌ Local file not found: {url}")

if img is None:
    print("❌ Failed to load image. Check the link/path.")
    exit()

# ======================
# Keep original for display
# ======================
orig_h, orig_w = img.shape[:2]
img_display = img.copy()

# ======================
# Run YOLO inference
# ======================
results = model(img, conf=0.2, iou=0.5, imgsz=min(orig_h, orig_w))[0]

# ======================
# Draw bounding boxes + labels
# ======================
for box in results.boxes:
    x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
    cls = int(box.cls[0])
    conf = float(box.conf[0])

    part_name = classNames[cls] if cls < len(classNames) else str(cls)
    color = colors[cls % len(colors)]

    # Bounding Box
    cvzone.cornerRect(img_display, (x1, y1, x2 - x1, y2 - y1), l=2, rt=2, colorR=color)

    # Label
    cvzone.putTextRect(
        img_display,
        f"{part_name} {conf:.2f}",
        (x1, max(y1 - 10, 10)),
        scale=1.0, thickness=2,
        colorR=color, colorT=(0, 0, 0),
        font=cv2.FONT_HERSHEY_SIMPLEX,
        offset=5, border=5, colorB=(255, 255, 255)
    )

# ======================
# Draw segmentation masks (with transparency)
# ======================
if results.masks is not None:
    alpha = 0.5  # transparency
    for seg, box in zip(results.masks.xy, results.boxes):
        cls = int(box.cls[0])
        color = colors[cls % len(colors)]

        pts = np.int32([seg])
        overlay = img_display.copy()
        cv2.fillPoly(overlay, pts, color)

        # Blend mask with original image
        img_display = cv2.addWeighted(overlay, alpha, img_display, 1 - alpha, 0)

        # Also draw polygon outline
        cv2.polylines(img_display, pts, True, color, 2)

# ======================
# Show in original resolution
# ======================
cv2.namedWindow("Detection Result", cv2.WINDOW_NORMAL)
cv2.imshow("Detection Result", img_display)
cv2.resizeWindow("Detection Result", orig_w, orig_h)
cv2.waitKey(0)
cv2.destroyAllWindows()


AttributeError: module 'cv2' has no attribute 'imshow'

In [ ]:
from flask import Flask, request, jsonify
from ultralytics import YOLO
import base64
import cv2
import numpy as np

# Load trained YOLO model
MODEL_PATH = r"D:\Himanshu_ML\TE_UAV Projects\TE_1 UAV MRO\UAV_Parts\runs\segment\Body_Annotation\weights\best.pt"
model = YOLO(MODEL_PATH)

app = Flask(__name__)

def base64_to_cv2(image_base64):
    """Convert base64 string to cv2 image"""
    img_data = base64.b64decode(image_base64)
    np_arr = np.frombuffer(img_data, np.uint8)
    img = cv2.imdecode(np_arr, cv2.IMREAD_COLOR)
    return img

@app.route("/predict", methods=["POST"])
def predict():
    data = request.json
    image_base64 = data["image"].split(",")[1]  # remove header
    img = base64_to_cv2(image_base64)

    # Run YOLO inference
    results = model(img, conf=0.25, iou=0.45)[0]

    predictions = []
    if results.masks is not None:
        for seg, box in zip(results.masks.xy, results.boxes):
            cls = int(box.cls[0])
            conf = float(box.conf[0])
            polygon = seg.tolist()

            predictions.append({
                "result": [{
                    "from_name": "label",
                    "to_name": "image",
                    "type": "polygonlabels",
                    "value": {
                        "points": polygon,
                        "polygonlabels": ["UAV"]  # <-- change if you have multiple classes
                    }
                }],
                "score": conf
            })

    return jsonify(predictions)

@app.route("/", methods=["GET"])
def health():
    return jsonify({"status": "YOLO Flask backend running ✅"})

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=9090, debug=True)


 * Serving Flask app "__main__" (lazy loading)
 * Environment: production
   Use a production WSGI server instead.
 * Debug mode: on


 * Restarting with watchdog (windowsapi)


SystemExit: 1

c:\Users\Himanshu\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
